# ECG Classification Using PTB-XL Dataset

## Dataset Overview

The **PTB-XL ECG Dataset** will be used to solve the ECG classification task. This comprehensive dataset includes:

- 21,799 clinical 12-lead ECGs
- Data collected from 18,869 patients
- Each recording is 10 seconds in length
- Sampling rate of 500 Hz

This large-scale public dataset is well-suited for supervised learning approaches to ECG classification, providing a robust foundation for developing diagnostic algorithms.

## Dataset Access

The PTB-XL dataset is publicly available through PhysioNet. For detailed information about the dataset structure, annotations, and download instructions, visit:
[PTB-XL Dataset on PhysioNet](https://physionet.org/content/ptb-xl/1.0.3/)

## Recommended Python Packages
For working with this ECG dataset, consider using these Python packages:
- `wfdb` - For reading and processing ECG signals
- `pandas` - For data manipulation and analysis
- `numpy` - For numerical operations
- `tqdm` - For monitoring progress

In [1]:
import functools
import pathlib
from pathlib import Path
import shutil
from typing import Union, Optional

import requests
from tqdm.auto import tqdm


def download_file(url: str, dest_filename: Union[str, Path]) -> Path:
    """
    Download a file from a URL and save it to the specified destination path.
    
    Args:
        url: The URL of the file to download.
        dest_filename: The destination path where the file will be saved.
        
    Returns:
        Path: The resolved path where the file was saved.
        
    Raises:
        requests.HTTPError: If the request returns a 4xx status code.
        RuntimeError: If the request returns any other non-200 status code.
    """
    # Make the request with streaming enabled
    request = requests.get(url, stream=True, allow_redirects=True)
    
    # Check if the request was successful
    if request.status_code != 200:
        request.raise_for_status()  # Raises HTTPError for 4xx status codes
        raise RuntimeError(f"Request to {url} returned status code {request.status_code}")
    
    # Get the file size if available
    file_size = int(request.headers.get('Content-Length', 0))

    # Prepare the destination path
    dest_path = Path(dest_filename).expanduser().resolve()  # expand ~ and ~user constructs
    dest_path.parent.mkdir(parents=True, exist_ok=True)  # Create parent directories if needed
    print(f"The file will be saved here: {dest_path}")

    # Prepare progress bar description
    desc = "(Unknown total file size)" if file_size == 0 else ""
    
    # Ensure proper decompression if needed
    request.raw.read = functools.partial(request.raw.read, decode_content=True)
    
    # Download the file with progress bar
    with tqdm.wrapattr(request.raw, "read", total=file_size, desc=desc) as r_raw:
        with dest_path.open("wb") as f:
            shutil.copyfileobj(r_raw, f)

    return dest_path

Make sure we need to download the PTB XL dataset via HTTP by its link. Note that the archive size is about 1.7 Gb.

In [2]:
import os
from typing import Optional

def ensure_ptb_xl_dataset(
    data_dir: str = "../data",
    dataset_url: Optional[str] = None
) -> str:
    """
    Ensures the PTB-XL ECG dataset is available locally, downloading it if necessary.
    
    Args:
        data_dir: Directory where the dataset should be stored
        dataset_url: URL to download the dataset from (if not provided, uses default URL)
    
    Returns:
        str: Path to the downloaded zip archive
    
    Note:
        The official resource page for this dataset is:
        https://physionet.org/content/ptb-xl/1.0.3/
    """
    # Use the provided URL or default to the official source
    if dataset_url is None:
        dataset_url = "https://physionet.org/static/published-projects/ptb-xl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip"
    
    # Create data directory if it doesn't exist
    os.makedirs(data_dir, exist_ok=True)
    
    # Define the destination path for the downloaded archive
    dest_archive = os.path.join(data_dir, "ptbxl.zip")
    
    # Download the file if it doesn't exist locally
    if not os.path.exists(dest_archive):
        download_file(dataset_url, dest_archive)
        
    return dest_archive

For furher processing we need only:
- the files from the records500 folder
- ptbxl_database.csv

Create a method to recursively collect file paths in a zip archive that match a specific pattern.

In [3]:
import os
import zipfile
import shutil
from zipfile import Path as ZipPath
from pathlib import Path
from typing import List, Set, Optional

def extract_from_zip(zip_file: str, paths: List[str], dest_folder: str, zip_folder: str = '') -> None:
    """
    Extract specific files and folders from a zip file to the destination folder.
    
    Args:
        zip_file: Path to the zip file
        paths: List of paths within the zip to extract
        dest_folder: Destination folder where files will be extracted
        zip_folder: Optional subfolder within the zip to extract from
        
    Returns:
        None
    """
    # Validate inputs
    if not Path(zip_file).exists() or not zipfile.is_zipfile(zip_file) or not paths:
        return
    
    # Create destination folder if it doesn't exist
    dest_path = Path(dest_folder)
    if not dest_path.exists():
        dest_path.mkdir(parents=True)

    # Prepare paths to extract
    selected_paths = [normalize_path(os.path.join(zip_file, zip_folder, path)) for path in paths]
    zip_files = []
    
    with zipfile.ZipFile(zip_file) as z:
        # Collect all files that match the selected paths
        for path in selected_paths:
            zip_files.extend(get_zip_files(ZipPath(z), path, dest_folder))
        
        # Extract all files at once (faster than individual extraction)
        all_zip_files = z.namelist()
        inner_zip_files = {file.replace(normalize_path(zip_file), "")[1:] for file in zip_files}
        files_to_remove = list(set(all_zip_files) - inner_zip_files)
        files_to_remove = [f"{dest_folder}/{file}" for file in files_to_remove]
        
        # Extract everything and then remove unwanted files
        shutil.unpack_archive(zip_file, dest_folder)

    # Clean up unwanted files
    for file_path in files_to_remove:
        if Path(file_path).exists():
            os.remove(file_path)


def get_zip_files(root: ZipPath, pattern: str, dest: str, flist: Optional[List[str]] = None) -> List[str]:
    """
    Recursively collect file paths in a zip archive that match a given pattern.
    
    Args:
        root: ZipPath object representing the current directory in the zip
        pattern: Pattern to match against file paths
        dest: Destination folder (used for reference only)
        flist: List to accumulate matching file paths
        
    Returns:
        List of file paths that match the pattern
    """
    if flist is None:
        flist = []
        
    for child in root.iterdir():
        str_child = normalize_path(child)
        if child.is_file() and str_child.startswith(pattern):
            flist.append(str_child)
        elif child.is_dir():
            # Continue pattern matching in subdirectories
            next_pattern = str_child if str_child.startswith(pattern) else pattern
            get_zip_files(child, next_pattern, dest, flist)

    return flist


def normalize_path(path: str) -> str:
    """
    Normalize path separators to forward slashes for consistent handling.
    
    Args:
        path: Path string to normalize
        
    Returns:
        Normalized path with forward slashes
    """
    return str(path).replace("\\", "/").replace("\\\\", "/")

Let's copy the target files from the archive with the declared methods. Note, that there are thousands binary files, so it may take some time to copy them.

In [4]:
import os
from pathlib import Path
import zipfile

# Dataset configuration
DATASET_NAME = "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"
ECG_RECORDS_FOLDER = "records500/"
ANNOTATIONS_FILE = "ptbxl_database.csv"

def extract_dataset(zip_path, data_dir):
    """
    Extract PTB-XL ECG dataset if not already extracted.
    
    Parameters:
    -----------
    zip_path : str or Path
        Path to the zip archive containing the dataset
    data_dir : str or Path
        Directory where the dataset should be extracted
    
    Returns:
    --------
    str
        Path to the extracted dataset
    """
    # Convert paths to Path objects for better cross-platform compatibility
    data_dir = Path(data_dir)
    zip_path = Path(zip_path)
    
    # Define the extraction destination
    extracted_folder = data_dir / DATASET_NAME
    ecg_data_path = extracted_folder / ECG_RECORDS_FOLDER
    
    # Check if data is already extracted
    if not ecg_data_path.exists():
        print("Starting data extraction...")
        
        # Create extraction directory if it doesn't exist
        extracted_folder.mkdir(exist_ok=True, parents=True)
        
        # Extract required files from the archive
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            files_to_extract = [
                f"{DATASET_NAME}/{ECG_RECORDS_FOLDER}",
                f"{DATASET_NAME}/{ANNOTATIONS_FILE}"
            ]
            for file in zip_ref.namelist():
                if any(file.startswith(prefix) for prefix in files_to_extract):
                    zip_ref.extract(file, data_dir)
        
        print(f"Data extracted to: {extracted_folder}")
    else:
        print(f"Data already extracted at: {extracted_folder}")
    
    return str(extracted_folder)

extracted_path = extract_dataset("../data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip", "../data")

Data already extracted at: ..\data\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3
